# 00 — Environment test

Environment checks only: validates Python, repository paths, required imports, a coordinate
transformation, library smoke tests, and output paths using generated test data.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import ensure_output_directories
ensure_output_directories()

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Personal\private\Portugal\Study\DataScience\Project\Reproducible_Wildfire_Exposure_Capstone


In [2]:
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import geopandas as gpd
import shapely
import pyproj
import pyogrio
import rasterio
import requests
import nbformat
import nbclient

packages = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit-learn": sklearn.__version__,
    "geopandas": gpd.__version__,
    "shapely": shapely.__version__,
    "pyproj": pyproj.__version__,
    "pyogrio": pyogrio.__version__,
    "rasterio": rasterio.__version__,
    "requests": requests.__version__,
    "nbformat": nbformat.__version__,
    "nbclient": nbclient.__version__,
}
pd.DataFrame(packages.items(), columns=["package", "version"])

,package,version
0,numpy,2.3.5
1,pandas,2.2.3
2,matplotlib,3.10.8
3,scikit-learn,1.8.0
4,geopandas,1.1.2
5,shapely,2.1.2
6,pyproj,3.7.2
7,pyogrio,0.12.1
8,rasterio,1.5.0
9,requests,2.32.5


In [3]:
from shapely.geometry import Point
from src.config import SPATIAL

sample = gpd.GeoDataFrame(
    {"location": ["Lisbon test point", "Porto test point"]},
    geometry=[Point(-9.1393, 38.7223), Point(-8.6291, 41.1579)],
    crs="EPSG:4326",
)
projected = sample.to_crs(SPATIAL.analysis_crs)

assert projected.crs.to_string() == SPATIAL.analysis_crs
assert projected.geometry.notna().all()
projected

,location,geometry
0,Lisbon test point,POINT (-87503.439 -104538.892)
1,Porto test point,POINT (-41630.673 165532.264)


In [4]:
from sklearn.linear_model import LogisticRegression

X = pd.DataFrame({
    "forest_shrub_share_2km": [0.10, 0.25, 0.70, 0.85, 0.45, 0.60],
    "fire_years_previous_10y_2km": [0, 0, 3, 5, 1, 2],
})
y = pd.Series([0, 0, 1, 1, 0, 1], name="burned_next_year")

model = LogisticRegression(random_state=42).fit(X, y)
probabilities = model.predict_proba(X)[:, 1]

assert probabilities.shape == (6,)
assert np.all((probabilities >= 0) & (probabilities <= 1))
probabilities

array([0.13180339, 0.13502228, 0.82380221, 0.97770557, 0.32854191,
       0.60310236])

In [5]:
from src.paths import FIGURES_DIR

figure_path = FIGURES_DIR / "00_environment_test.png"
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(X["forest_shrub_share_2km"], probabilities)
ax.set_xlabel("Forest and shrubland share within 2 km")
ax.set_ylabel("Test model probability")
ax.set_title("Environment test — generated data")
fig.tight_layout()
fig.savefig(figure_path, dpi=120)
plt.close(fig)

assert figure_path.exists()
print(f"Environment validation passed. Figure written to: {figure_path}")

Environment validation passed. Figure written to: C:\Personal\private\Portugal\Study\DataScience\Project\Reproducible_Wildfire_Exposure_Capstone\reports\figures\00_environment_test.png
